<a href="https://colab.research.google.com/github/arman-h-alavi/thesis_humanizer/blob/main/llama3_fp16_successful_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Configure 4-bit Quantization
print("\nConfiguring 4-bit Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

MODEL_ID = "NousResearch/Meta-Llama-3-8B"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Loading Base Model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Attaching LoRA Adapter...")
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)

print("Loading Dataset...")
dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/Thesis_Humanizer/train_dataset.jsonl",
    split="train"
)

# ---------------------------------------------------------
print("Configuring Training Arguments (Bypassing AMP)...")
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/Thesis_Humanizer/checkpoints",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    num_train_epochs=1,
    fp16=False,                         # FIX: Completely disable buggy AMP
    bf16=False,                         # FIX: Ensure BFloat16 is off
    max_grad_norm=0.3,
    warmup_steps=10,
    save_strategy="epoch",
)
# ---------------------------------------------------------

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("\n🚀 Starting Training... (Leave this browser tab open!)")
trainer.train()

print("\n💾 Training Complete! Saving your custom LoRA adapter to Drive...")
trainer.model.save_pretrained("/content/drive/MyDrive/Thesis_Humanizer/final_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/Thesis_Humanizer/final_adapter")
print("✅ All done. Your academic humanizer is ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Configuring 4-bit Quantization...
Loading Tokenizer...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Loading Base Model...
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Attaching LoRA Adapter...
Loading Dataset...
Configuring Training Arguments (Bypassing AMP)...
Initializing SFTTrainer...
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.

🚀 Starting Training... (Leave this browser tab open!)

<div>
      
      <progress value='246' max='246' style='width:300px; height:20px; vertical-align: middle;'></progress>
      [246/246 56:37, Epoch 1/1]
    </div>
    <table border="1" class="dataframe">
  <thead>
 <tr style="text-align: left;">
      <th>Step</th>
      <th>Training Loss</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>10</td>
      <td>2.516947</td>
    </tr>
    <tr>
      <td>20</td>
      <td>2.430925</td>
    </tr>
    <tr>
      <td>30</td>
      <td>2.403652</td>
    </tr>
    <tr>
      <td>40</td>
      <td>2.325708</td>
    </tr>
    <tr>
      <td>50</td>
      <td>2.360431</td>
    </tr>
    <tr>
      <td>60</td>
      <td>2.362058</td>
    </tr>
    <tr>
      <td>70</td>
      <td>2.281446</td>
    </tr>
    <tr>
      <td>80</td>
      <td>2.357283</td>
    </tr>
    <tr>
      <td>90</td>
      <td>2.282521</td>
    </tr>
    <tr>
      <td>100</td>
      <td>2.350148</td>
    </tr>
    <tr>
      <td>110</td>
      <td>2.348013</td>
    </tr>
    <tr>
      <td>120</td>
      <td>2.375573</td>
    </tr>
    <tr>
      <td>130</td>
      <td>2.325801</td>
    </tr>
    <tr>
      <td>140</td>
      <td>2.305063</td>
    </tr>
    <tr>
      <td>150</td>
      <td>2.310167</td>
    </tr>
    <tr>
      <td>160</td>
      <td>2.339458</td>
    </tr>
    <tr>
      <td>170</td>
      <td>2.252985</td>
    </tr>
    <tr>
      <td>180</td>
      <td>2.243744</td>
    </tr>
    <tr>
      <td>190</td>
      <td>2.368967</td>
    </tr>
    <tr>
      <td>200</td>
      <td>2.234977</td>
    </tr>
    <tr>
      <td>210</td>
      <td>2.276842</td>
    </tr>
    <tr>
      <td>220</td>
      <td>2.254805</td>
    </tr>
    <tr>
      <td>230</td>
      <td>2.304274</td>
    </tr>
    <tr>
      <td>240</td>
      <td>2.314833</td>
    </tr>
  </tbody>
</table><p>